In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# MIMIC-IV: ALL SEPSIS ICU STAYS
# SAE STAYS ARE LABELLED, NOT FILTERED OUT
# ============================================================

ROOT = Path("/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning")
MIMIC = ROOT / "data" / "raw" / "mimic-iv"
HOSP = MIMIC / "hosp"
ICU = MIMIC / "icu"

OUT = ROOT / "sae_mimiciv_v2" / "data"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD CORE TABLES
# ------------------------------------------------------------

diagnoses = pd.read_csv(HOSP / "diagnoses_icd.csv.gz")
d_icd = pd.read_csv(HOSP / "d_icd_diagnoses.csv.gz")
icustays = pd.read_csv(ICU / "icustays.csv.gz")

# ------------------------------------------------------------
# 2. IDENTIFY SEPSIS AND ENCEPHALOPATHY
# ------------------------------------------------------------

text_cols = [
    c for c in ["long_title", "short_title"]
    if c in d_icd.columns
]

sepsis_mask = False
enceph_mask = False

for c in text_cols:
    text = d_icd[c].astype(str)

    sepsis_mask = sepsis_mask | text.str.contains(
        "sepsis|septic",
        case=False,
        na=False,
        regex=True
    )

    enceph_mask = enceph_mask | text.str.contains(
        "encephal",
        case=False,
        na=False,
        regex=True
    )

sepsis_codes = d_icd.loc[
    sepsis_mask,
    ["icd_code", "icd_version"]
].drop_duplicates()

enceph_codes = d_icd.loc[
    enceph_mask,
    ["icd_code", "icd_version"]
].drop_duplicates()

# ------------------------------------------------------------
# 3. FIND SEPSIS ADMISSIONS
# ------------------------------------------------------------

sepsis_dx = diagnoses.merge(
    sepsis_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

sepsis_hadm = set(
    sepsis_dx["hadm_id"]
)

# ------------------------------------------------------------
# 4. FIND ENCEPHALOPATHY ADMISSIONS
# ------------------------------------------------------------

enceph_dx = diagnoses.merge(
    enceph_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

enceph_hadm = set(
    enceph_dx["hadm_id"]
)

# SAE = SEPSIS + ENCEPHALOPATHY
sae_hadm = sepsis_hadm.intersection(
    enceph_hadm
)

# ------------------------------------------------------------
# 5. ALL SEPSIS ICU STAYS
# ------------------------------------------------------------

sepsis_icu = icustays[
    icustays["hadm_id"].isin(sepsis_hadm)
].copy()

sepsis_icu["intime"] = pd.to_datetime(
    sepsis_icu["intime"]
)

sepsis_icu["outtime"] = pd.to_datetime(
    sepsis_icu["outtime"]
)

# Mark SAE ICU stays
sepsis_icu["sae"] = (
    sepsis_icu["hadm_id"]
    .isin(sae_hadm)
    .astype(int)
)

print("=" * 60)
print("SEPSIS ICU COHORT")
print("=" * 60)

print(
    "Sepsis admissions:",
    len(sepsis_hadm)
)

print(
    "Sepsis patients:",
    sepsis_icu["subject_id"].nunique()
)

print(
    "Sepsis ICU stays:",
    sepsis_icu["stay_id"].nunique()
)

print(
    "SAE admissions:",
    len(sae_hadm)
)

print(
    "SAE patients:",
    sepsis_icu.loc[
        sepsis_icu["sae"] == 1,
        "subject_id"
    ].nunique()
)

print(
    "SAE ICU stays:",
    sepsis_icu.loc[
        sepsis_icu["sae"] == 1,
        "stay_id"
    ].nunique()
)

print("\nStay-level label distribution:")
print(
    sepsis_icu["sae"].value_counts()
)

# ------------------------------------------------------------
# 6. CREATE COMPLETE HOURLY GRID
# ------------------------------------------------------------

hourly_parts = []

for _, stay in sepsis_icu.iterrows():

    start = stay["intime"].floor("h")
    end = stay["outtime"].floor("h")

    hours = pd.date_range(
        start=start,
        end=end,
        freq="h"
    )

    temp = pd.DataFrame({
        "subject_id": stay["subject_id"],
        "hadm_id": stay["hadm_id"],
        "stay_id": stay["stay_id"],
        "hour_time": hours,
        "sae": stay["sae"]
    })

    hourly_parts.append(temp)

hourly = pd.concat(
    hourly_parts,
    ignore_index=True
)

# ICU-relative hour
icu_start = (
    hourly
    .groupby("stay_id")["hour_time"]
    .transform("min")
)

hourly["hour"] = (
    (
        hourly["hour_time"] - icu_start
    ).dt.total_seconds() / 3600
).astype(int)

# ------------------------------------------------------------
# 7. SAVE BASE HOURLY DATASET
# ------------------------------------------------------------

output_file = (
    OUT / "sepsis_all_hourly_base.csv"
)

hourly.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 60)
print("BASE HOURLY DATASET CREATED")
print("=" * 60)

print(
    "Rows:",
    len(hourly)
)

print(
    "Patients:",
    hourly["subject_id"].nunique()
)

print(
    "ICU stays:",
    hourly["stay_id"].nunique()
)

print(
    "SAE hourly rows:",
    int(hourly["sae"].sum())
)

print(
    "Non-SAE hourly rows:",
    int((hourly["sae"] == 0).sum())
)

print(
    "\nSaved:",
    output_file
)

print("\nStay-level summary:")
display(
    sepsis_icu[
        [
            "subject_id",
            "hadm_id",
            "stay_id",
            "intime",
            "outtime",
            "sae"
        ]
    ].sort_values("sae", ascending=False)
)

SEPSIS ICU COHORT
Sepsis admissions: 24
Sepsis patients: 17
Sepsis ICU stays: 26
SAE admissions: 6
SAE patients: 6
SAE ICU stays: 9

Stay-level label distribution:
sae
0    17
1     9
Name: count, dtype: int64

BASE HOURLY DATASET CREATED
Rows: 3769
Patients: 17
ICU stays: 26
SAE hourly rows: 1517
Non-SAE hourly rows: 2252

Saved: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_mimiciv_v2/data/sepsis_all_hourly_base.csv

Stay-level summary:


,subject_id,hadm_id,stay_id,intime,outtime,sae
90,10031757,28477280,30458995,2137-10-12 22:44:57,2137-10-14 17:08:34,1
53,10002428,28662225,33987268,2156-04-12 16:24:18,2156-04-17 15:57:08,1
32,10003400,23559586,38383343,2137-08-17 17:36:37,2137-09-02 19:17:11,1
33,10004235,24181354,34100191,2196-02-24 17:07:00,2196-02-29 15:58:02,1
82,10031757,28477280,33244906,2137-10-15 17:29:21,2137-10-17 22:16:51,1
37,10020944,29974575,30757476,2131-02-27 16:40:00,2131-03-08 18:30:38,1
38,10002428,28662225,38875437,2156-04-19 18:11:19,2156-04-26 18:58:41,1
9,10018081,21027282,37293400,2133-12-18 17:10:00,2134-01-01 14:44:53,1
131,10003400,23559586,34577403,2137-08-10 19:54:51,2137-08-13 17:54:54,1
114,10037861,24540843,34531557,2117-03-14 16:34:58,2117-03-25 02:35:08,0
